In [19]:
from dotenv import load_dotenv
from crewai_tools import SerperDevTool
from crewai import Agent, Task, Crew, Process

#load_dotenv()

#tool = SerperDevTool(n_results = 1)
#response = tool.run(search_query = "한국 금리 전망")
#print(response)

# n_results 를 설정해 놓아서 1개의 결과가 나옴. 기본값은 10임. 검색 옵션을 지정해 좀더 지역화된 겨로가를 얻을 수 있음.
# 예를 들어, 특정 국가의 검색 결과만 보거나 학술 검색 API 엔드포인트를 사용할 수도 있음.
#tools = SerperDevTool(
#    country = "kr",
#    locale = "ko",
#    n_results = 1
#)

#print(tools.run(search_query="부동산 시장 2025년 전망"))

In [20]:
# 01. SerperDev 도구 정의
# SerperDevTool 를 초기화해서 웹 검색 기능을 수행할 수 있도록 구성함.
# n_results 값을 통해 가져올 검색 결과 개수를 지정할 수 있으며, 사전에 발급받아 환경 변수로 등록한 SERPER_API_KEY 를 기반으로  Server API 를 호출하게 함.

#1)  SerperDevTool 정의(웹 검색 전용 도구)
serper_tool = SerperDevTool(
    n_results = 5 # 상위 5개 결과만 사용
)

In [21]:
# 02. 에이전트 정의
# real_estate_research 는 SerperDevTool 를 활용돼 웹 검색을 수행하는 역할을 담당.
# tools = [serper_tool] 로 설정했기 때문에 자연어 프롬프트 내부에서 웹 검색을 직접 수행할 수 있음.
# report_writer는 검색된 내용을 바탕으로 종합 분석 리포트를 작성하는 역할로, 사용자에게 읽기 쉬운 내용을 구성하는데 집중함.

#2) 에이전트 정의
# 2-1) 웹에서 자료를 찾는 리서치 에이전트
real_estate_researcher = Agent(
    role = "부동산 리서치 에이전트",
    goal = (
        "웹 검색을 통해 최신 한국 부동산 시장 동향과 전망을 찾아"
        "핵심 정보만 정리한다."
    ),
    backstory = (
        "각종 뉴스, 리포트, 컬럼을 분석해 요약해 둔"
        "온라인 리서치 전문가이다"
    ),
    tools = [serper_tool], # SerperDevTool 연결
    verbose = True
)

# 2-2) 리포트 형태로 정리하는 에이전트
report_writer = Agent(
    role = "부동산 리포트 작성 에이전트",
    goal = "수집된 정보를 바탕으로 이해하기 쉬운 한국어 시장 전망 리포트를 작성한다.",
    backstory = "경제/부동산 관련 리포트를 다수 작성해 둔 분석가이다",
    verbose = True
)

In [22]:
# 03. 작업 정의
# search_task 는 먼저 Serper 검색을 실행해 관련 기사를 탐색하고 핵심 내용을 정리하는 단계임.
# expected_output 을 통해 어떤 구조로 요약해야 하는지가 명확히 정의돼 있으며, 후속 단계에서 바로 활용할 수 있음.
# analysis_task 는 search_task의 결과를 입력으로 ㅂ다아 부동산 시장의 단기, 중기 전망을 분석 보고서 형태로 작성함.

# 3) 테스크 정의
# 3-1) 웹에서 정보 수집
search_task = Task(
    description = (
        "다음 질문에 대해 SerperDevTool을 사용해 웹을 검색하고,"
        "관련성이 높은 기사와 리포트를 찾아 요약하라 \n"
        "질문 : {question}\n\n"
        "요구사항 : \n"
        "1) 최근 1~2년 내 기사/보고서를 우선적으로 참고한다.\n"
        "2) 서로 다른 출처(언론사, 리포트)를 최소 2개 이상 포함한다. \n"
    ),
    expected_output = (
        "다음 형식의 요약 노드를 작성한다. \n\n"
        "1. 참고한 주요 기사/ 리포트 목록 (제목, 출처, URL) \n"
        "2. 기사/리포트에서 공통적으로 언급하는 핵심 키워드 3~5개\n"
        "3. 각 키워드에 대한 한두 문장 요약"
    ),
    agent = real_estate_researcher,
    tools = [serper_tool],
    verbose = True
)

# 3-2) 수집된 내용을 바탕으로 전망 리포트를 작성.
analysis_task = Task(
    description = (
        "이전 테스크에서 정리한 웹 검색 요약을 바탕으로"
        "한국 부동산 시장 전망에 대해 분석 리포트를 작성하라. \n"
        "질문 : {question}\n\n"
        "요구사항 : \n"
        "1) 단기(1년 이내)와 중기(3년 이내) 전망을 나누어 서술한다. \n"
        "2) 상승/하락 요인과 리스트 요인을 구분해 정리한다.\n"
        "3) 과도한 확신 표현은 피히고, '가능성이 높다/낮다' 수준으로 표현한다.\n"
    ),
    expected_output = (
        "다음 구조의 한국어 리포트를 작성한다.\n\n"
        "1. 한줄 요약\n"
        "2. 단기 전망 (1년 이내)\n"
        "3. 중기 전망 (3년 이내)\n"
        "4. 주요 상승 요인\n"
        "5. 주요 하락, 리스크 요인\n"
        "6. 참고: 사용한 기사/리포트와 간단한 출력 요약"
    ),
    agent = report_writer,
    context = [search_task], # 검색 -> 간단한 출력 요약.
    verbose = True,
)

In [ ]:
# 04. 크루 구성
real_estate_crew = Crew(
    agents = [real_estate_researcher, report_writer],
    tasks = [search_task, analysis_task],
    process = Process.sequential,
    verbose = True,
)

In [ ]:
# 05. 실행
# 실행 결과를 보면 검색 태스크 크기 SerperDevTool를 통해 관련 기사들을 수집하고 핵심 키워드를 구조화해 제공했으며, 분석 태스크는 이를 기반으로 단기, 중기 전망과 상승, 하락 요인을 포함한 완성도 높은 부동산 시장 리포트를 작성했음.

question = "2026년 향후 한국 부동산 시장 전망과 주요 리스크는 무엇인가?"

result = await real_estate_crew.kickoff_async(inputs = {"question": question})

print("\n=== Crew 통합 결과(result.raw) ===")
print(result.raw)

print("\n=== 검색 태스크(search_task) 출력 ===")
print(search_task.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8facd09d-70eb-4535-8b7d-859c98b6d3c3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 다음 질문에 대해 SerperDevTool을 사용해 웹을 검색하고,관련성이 높은 기사와 리포트를 찾아 요약하라        │
│  질문 : 2026년 향후 한국 부동산 시장 전망과 주요 리스크는 무엇인가?                                             │
│                                                                                                                 │
│  요구사항 :                                                                                                     │
│  1) 최근 1~2년 내 기사/보고서를 우선적으로 참고한다.                                                            │
│  2) 서로 다른 출처(언론사, 리포트)를 최소 2개 이상 포함한다.                                                    │
│                                                                                                                 │
│  Agent: 부동산 리서치 에이전트                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 다음 질문에 대해 SerperDevTool을 사용해 웹을 검색하고,관련성이 높은 기사와 리포트를 찾아 요약하라        │
│  질문 : 2026년 향후 한국 부동산 시장 전망과 주요 리스크는 무엇인가?                                             │
│                                                                                                                 │
│  요구사항 :                                                                                                     │
│  1) 최근 1~2년 내 기사/보고서를 우선적으로 참고한다.                                                            │
│  2) 서로 다른 출처(언론사, 리포트)를 최소 2개 이상 포함한다.                                                    │
│                                                                                                                 │
│  ID: e62e99a2-3fb4-4d87-aaf0-9cd29fcb8c1a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 8facd09d-70eb-4535-8b7d-859c98b6d3c3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯